# 1.5B **policy** — staged PPO reusing the 0.8025 reward model

The absolute-cleanest stretch result (HANDOFF #3): RL a **Qwen2.5-1.5B-Instruct** policy (LoRA) 
against the already-trained **0.8025** 1.5B reward model. RM is *not* retrained here — a full 
1.5B RM+PPO does not fit one 12 h session, so this stage is PPO-only (~5 h on a T4).

**ONE manual prerequisite:** attach the 0.8025 RM as a Kaggle **Dataset** (`+ Add Input`). Create 
it once via *New Dataset -> from the 1.5B-RM kernel output* (`checkpoints/reward_model/`), because 
local CLI downloads of that checkpoint come back partial. This notebook auto-discovers it under 
`/kaggle/input` (marker: `reward_config.json`) and asserts the merged weights are non-empty.

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python scripts/smoke_test.py 2>&1 | tail -2

## Config + locate the attached 0.8025 reward model

In [ ]:
import glob, os, shutil, json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-1.5B-Instruct'
DATA_CLEAN='argilla/ultrafeedback-binarized-preferences-cleaned'
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
# Find the RM checkpoint from any attached Dataset (marker file written by RewardModel.save_pretrained).
hits=glob.glob('/kaggle/input/**/reward_config.json', recursive=True)
assert hits, ('No reward model under /kaggle/input. Attach the 0.8025 RM as a Dataset: + Add Input '
              '-> the dataset you created from the 1.5B-RM kernel output (checkpoints/reward_model/).')
RM_SRC=os.path.dirname(sorted(hits)[0])
os.makedirs('checkpoints', exist_ok=True)
if os.path.abspath(RM_SRC)!=os.path.abspath('checkpoints/reward_model'):
    shutil.rmtree('checkpoints/reward_model', ignore_errors=True)
    shutil.copytree(RM_SRC, 'checkpoints/reward_model')
wt=glob.glob('checkpoints/reward_model/*.safetensors')+glob.glob('checkpoints/reward_model/*.bin')
sz=max([os.path.getsize(f) for f in wt], default=0)
print(f'RM <- {RM_SRC} ({sz/1e6:.0f} MB weights); policy={MODEL} dtype={DTYPE}')
assert sz>1e7, 'RM weights empty/partial — re-create the Dataset from a COMPLETE kernel output.'

## PPO — 1.5B LoRA policy, RL against the 0.8025 reward model (~5 h)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=$MODEL -o policy.dtype=$DTYPE -o policy.use_lora=true \
  -o reward_model.name_or_path=checkpoints/reward_model -o reward_model.dtype=$DTYPE \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.max_samples=4000 \
  -o ppo.total_episodes=1024 -o ppo.rollout_batch_size=8 -o ppo.mini_batch_size=1 \
  -o ppo.lr=1e-6 -o ppo.normalize_rewards=true -o ppo.save_every=200 \
  -o generation.max_new_tokens=40 -o data.max_prompt_length=160
# OOM? drop -o ppo.rollout_batch_size=4 (1.5B LoRA policy + value head + frozen 1.5B RM on one T4 is tight).

## Evaluate — PPO vs base Instruct, RM-judged win-rate -> RESULTS.md

In [ ]:
import subprocess
def run(c):
    print('$',c,flush=True); o=subprocess.run(c,shell=True,capture_output=True,text=True)
    print(o.stdout[-700:])
    if o.returncode: print(o.stderr[-1200:])
    return o.stdout
wn=run(f"python scripts/evaluate.py score-policy --policy checkpoints/ppo "
        f"--reward-model checkpoints/reward_model --compare {MODEL} --data {DATA_CLEAN} "
        f"--split 'train[2000:]' --num 100 --max-new-tokens 40 --max-length 320")
md=(f'# 1.5B PPO policy results\n\n- policy: `{MODEL}` (LoRA, PPO) vs base `{MODEL}`\n'
    f'- reward model: the 0.8025 1.5B RM (reused, not retrained)\n\n'
    f'## PPO vs Instruct - RM-judged win-rate + samples\n```\n{wn}\n```\n\n'
    f'_Independent Claude-judge win-rate: run locally after download (key in .env)._\n')
open('/kaggle/working/RESULTS.md','w').write(md); print('\nSaved -> RESULTS.md')

### Read
Win-rate > 50% means PPO improved the 1.5B policy under its own RM (Goodhart-circular — confirm with 
the independent Claude judge locally: `scripts/evaluate.py judge --policy <ppo> --base Qwen/Qwen2.5-1.5B-Instruct --num 100 --device cpu`).